In [0]:
from pyspark.sql import functions as F, Window
from pyspark.ml.feature import VectorAssembler

In [0]:
# Fixed training window: Oct – Nov 2025 (fault-dense, stable period)
# Extended to 2 months to allow for 15-day gap in train/test split
start_date = "2025-10-01"
end_date = "2025-12-01"

df_sx_telemetry = spark.table('marel_digital_machines.sensorx.marel_sensorx_telemetry')
df_sx_telemetry = df_sx_telemetry.filter(
    (F.col('timeStamp') >= start_date) &
    (F.col('timeStamp') < end_date)
).select(
    "timeStamp",
    "properties_deviceId",
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "payload_xrayController_onTime"
)

In [0]:
# Read csv with generator info (500W / 800W)
df_csv = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .option("quote", "\"") \
    .option("delimiter", ",") \
    .load("/Volumes/teams/sensorx/data-dump/deviceID_Serial_GenSerial_machineCode.csv")

# Device IDs that have at least one 800W serial number
devices_800w = (
    df_csv.filter(F.col("gentype") == "800W")
    .select("properties_deviceId")
    .distinct()
)

# Filter telemetry to only 800W devices
df_sx_telemetry = df_sx_telemetry.join(
    F.broadcast(devices_800w), "properties_deviceId", "inner"
)

print("Filtered telemetry to 800W devices")

In [0]:
df_sx_xraycontroller_fault = (
    spark.table('marel_digital_machines.sensorx.marel_sensorx_xraycontroller_property_fault')
    .filter(
        (F.col('timeStamp') >= start_date) &
        (F.col('timeStamp') < end_date) &
        (F.col('payload_fault_faultName') == "faultRegulation")
    )
    .select("timeStamp", "properties_deviceId", "payload_fault_faultName", "payload_fault_state")
)

# Filter faults to 800W devices only
df_sx_xraycontroller_fault = df_sx_xraycontroller_fault.join(
    F.broadcast(devices_800w), "properties_deviceId", "inner"
)

print("Fault data filtered to 800W devices")

In [0]:
# --- Union + forward-fill ---
# Stack telemetry and fault data, sort by time, propagate fault state forward.

# Telemetry columns not shared with fault data
telem_only = [c for c in df_sx_telemetry.columns
              if c not in ("timeStamp", "properties_deviceId")]

# Telemetry rows with null fault placeholder
telem_part = df_sx_telemetry.select(
    "timeStamp", "properties_deviceId",
    *telem_only,
    F.lit(None).cast("boolean").alias("payload_fault_state"),
    F.lit(True).alias("_is_telem"),
)

# Fault rows with null telemetry placeholders
fault_part = df_sx_xraycontroller_fault.select(
    "timeStamp", "properties_deviceId",
    *[F.lit(None).cast(dict(df_sx_telemetry.dtypes)[c]).alias(c)
      for c in telem_only],
    "payload_fault_state",
    F.lit(False).alias("_is_telem"),
)

# Union, forward-fill fault state per device, keep telemetry rows only
w = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

df_sx_joined_regu_fault_800 = (
    telem_part.unionByName(fault_part)
    .withColumn("payload_fault_state",
                F.last("payload_fault_state", ignorenulls=True).over(w))
    .filter(F.col("_is_telem"))
    .drop("_is_telem")
)

# Default nulls: rows before the first fault event get False
df_sx_joined_regu_fault_800 = df_sx_joined_regu_fault_800.withColumn(
    "payload_fault_state", F.coalesce(F.col("payload_fault_state"), F.lit(False))
)

print("Union + forward-fill complete")

In [0]:
# Adding delta seconds between consecutive readings per device
df = df_sx_joined_regu_fault_800

w = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

df = (
    df
    .withColumn("prev_ts", F.lag("timeStamp").over(w))
    .withColumn(
        "delta_seconds",
        F.when(
            F.col("prev_ts").isNull() |
            (F.col("timeStamp").cast("long") - F.col("prev_ts").cast("long") < 0),
            None
        ).otherwise(
            F.col("timeStamp").cast("long") - F.col("prev_ts").cast("long")
        ).cast("double")
    )
    .drop("prev_ts")
)

In [0]:
# Explicit sensor columns
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_onTime",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "delta_seconds",
]

df = df.na.drop(subset=sensor_cols)

# Encode properties_deviceId as integer index
device_ids = df.select("properties_deviceId").distinct().orderBy("properties_deviceId")
device_ids = device_ids.withColumn("deviceId_index", (F.dense_rank().over(Window.orderBy("properties_deviceId")) - 1).cast("double"))
df_encoded = df.join(device_ids, on="properties_deviceId", how="inner")

# Horizon (binary: failure within H_days or not)
H_days = 15
H_seconds = H_days * 86400

w_horizon = (
    Window.partitionBy("properties_deviceId")
    .orderBy(F.col("timeStamp").cast("long"))
    .rangeBetween(0, H_seconds)
)

df_encoded = df_encoded.withColumn(
    "failure_horizon",
    F.max(F.col("payload_fault_state").cast("int")).over(w_horizon)
)

# Lag features (n_lags = 20)
n_lags = 20
w = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

lag_exprs = [F.lag(col_name, L).over(w).alias(f"{col_name}{L}")
             for L in range(1, n_lags + 1) for col_name in sensor_cols]
df_encoded = df_encoded.select("*", *lag_exprs)

lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

# Break lineage via Delta temp path (memory-efficient, no named table)
temp_path = "/tmp/sensorx_lag_prep"
df_encoded.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(temp_path)
df_encoded = spark.read.format("delta").load(temp_path)

# Deviation features: 24-hour time-based rolling average
w_daily = (
    Window.partitionBy("properties_deviceId")
    .orderBy(F.col("timeStamp").cast("long"))
    .rangeBetween(-86400, 0)
)

dev_sensors = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
]
dev_cols = []
dev_exprs = []

for col_name in dev_sensors:
    avg_col = f"{col_name}_avg_daily"
    dev_col = f"{col_name}_dev_daily"
    dev_cols.extend([avg_col, dev_col])
    dev_exprs.append(F.avg(col_name).over(w_daily).alias(avg_col))
    dev_exprs.append((F.col(col_name) - F.avg(col_name).over(w_daily)).alias(dev_col))

df_encoded = df_encoded.select("*", *dev_exprs)

print(f"Added {len(dev_cols)} deviation features (24-hour window)")

# Drop nulls and assemble
df_clean = df_encoded.na.drop(subset=lag_cols + dev_cols)

feature_input_cols = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features")
df_features = assembler.transform(df_clean)

df_features = df_features.select("timeStamp", "features", F.col("failure_horizon").alias("label"))

# Cache to avoid recomputation during multiple actions below
df_features = df_features.cache()
row_count = df_features.count()
print(f"\nTotal rows after feature engineering: {row_count:,}")

if row_count > 0:
    print(f"Feature vector size: {df_features.select('features').first()[0].size}")
    print(f"  - {len(sensor_cols)} sensor cols")
    print(f"  - {len(lag_cols)} lag cols")
    print(f"  - {len(dev_cols)} deviation cols (24-hour window)")
    print(f"  - 1 deviceId index")
    print(f"\nLabel distribution (H={H_days} days):")
    df_features.groupBy("label").count().orderBy("label").show()
else:
    print("WARNING: No rows survived feature engineering! Check upstream data.")

# --- Chronological train/test split with 15-day gap (prevents horizon leakage) ---
# Split point at 70th percentile (leaves room for gap + test)
cutoff_epoch = df_features.selectExpr(
    "percentile_approx(CAST(timeStamp AS BIGINT), 0.7)"
).first()[0]

# Max timestamp in data
max_epoch = df_features.selectExpr(
    "MAX(CAST(timeStamp AS BIGINT))"
).first()[0]

gap_seconds = H_seconds  # 15 days

# Train: all readings before (split_point - gap)
# Their 15-day horizon labels only reference events before the split point
train = df_features.filter(F.col("timeStamp").cast("bigint") <= cutoff_epoch - gap_seconds)

# Test: readings after split point, excluding last 15 days (incomplete horizon labels)
test = df_features.filter(
    (F.col("timeStamp").cast("bigint") > cutoff_epoch) &
    (F.col("timeStamp").cast("bigint") <= max_epoch - gap_seconds)
)

train_count = train.count()
test_count = test.count()
print(f"\nTrain/test split with {H_days}-day gap:")
print(f"  Train: {train_count:,} rows")
print(f"  Test:  {test_count:,} rows")
print(f"  Gap:   {H_days} days dropped (prevents horizon leakage)")
print(f"  Test end trimmed: last {H_days} days dropped (incomplete labels)")

# Verify label distributions are consistent
print(f"\nTrain label distribution:")
train.groupBy("label").count().orderBy("label").show()
print(f"Test label distribution:")
test.groupBy("label").count().orderBy("label").show()

In [0]:
train.write.mode("overwrite").saveAsTable("teams.sensorx.train_data")
test.write.mode("overwrite").saveAsTable("teams.sensorx.test_data")